# Demonstração: LLM com prompting e RAG simples

Nesta demonstração, usamos um LLM pequeno para mostrar como prompting, parâmetros de amostragem e RAG funcionam na prática. O foco é didático: entender o fluxo e as limitações, não obter o melhor desempenho possível.

## 1. Importações e configuração
Aqui carregamos as bibliotecas necessárias e definimos uma semente para reprodutibilidade.

In [1]:
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

SEED = 42
np.random.seed(SEED)
set_seed(SEED)

/media/rodolfo/Cold Files/Projects/Github/unimar/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Carregar tokenizer e modelo
Usamos um modelo pequeno em português com ajuste por instrução apenas para fins didáticos. O mesmo fluxo funciona para modelos maiores.

In [2]:
model_name = 'Polygl0t/Tucano2-qwen-0.5B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 9726.90it/s]


## 3. Tokenização básica
LLMs operam em tokens. Vamos ver como o texto é transformado em tokens.

In [3]:
text_example = "IA generativa aprende padrões e gera novos textos."
inputs = tokenizer(text_example, return_tensors='pt')

tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
print('Texto:', text_example)
print('Número de tokens:', len(tokens))
print('Tokens:', tokens[:15])

Texto: IA generativa aprende padrões e gera novos textos.
Número de tokens: 10
Tokens: ['IA', '▁gener', 'ativa', '▁aprende', '▁padrões', '▁e', '▁gera', '▁novos', '▁textos', '.']


## 4. Função utilitária de geração
Criamos uma função simples para gerar texto com controle de temperatura e top-p.

In [4]:
def generate_text(prompt, max_new_tokens=80, temperature=0.7, top_p=0.9):
    set_seed(SEED)
    encoded = tokenizer(prompt, return_tensors='pt').to(device)
    outputs = model.generate(
        **encoded,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

## 5. Geração zero-shot
Um prompt direto já produz texto, mas a resposta pode conter erros ou imprecisões.

In [5]:
prompt = "Explique em duas frases o que é um LLM."
print(generate_text(prompt, max_new_tokens=60, temperature=0.7, top_p=0.9))

Explique em duas frases o que é um LLM.
Um mestrado (MSc.) ou doutorado (PhD.), também conhecido como mestrado e doutorado, respectivamente, são programas de pós-graduação projetados para indivíduos com experiência profissional significativa em sua área escolhida. Esses graus avançados podem ser obtidos por meio do programa acadêmico tradicional ou através da formação especializada avançada, permitindo


## 6. Efeito de temperatura e top-p
Temperatura controla aleatoriedade. Valores mais altos geram respostas mais variadas.

In [6]:
prompt = "Escreva um parágrafo curto sobre IA na educação."

for temp in [0.2, 0.7, 1.0]:
    print('Temperatura:', temp)
    print(generate_text(prompt, max_new_tokens=70, temperature=temp, top_p=0.9))

Temperatura: 0.2
Escreva um parágrafo curto sobre IA na educação.
A inteligência artificial (IA) está sendo cada vez mais utilizada no setor da educação para melhorar a experiência de aprendizagem dos alunos, automatizar tarefas administrativas e fornecer feedback personalizado aos professores. A tecnologia AI pode analisar grandes quantidades de dados educacionais para identificar padrões e tendências que podem informar as decisões pedagógicas e ajudar os educadores a adaptar suas abordagens às necessidades individuais
Temperatura: 0.7
Escreva um parágrafo curto sobre IA na educação.
A inteligência artificial (IA) tornou-se uma ferramenta indispensável no campo da educação, revolucionando a forma como os professores interagem com os alunos e proporcionando experiências de aprendizagem personalizadas para todos. A tecnologia AI pode analisar grandes quantidades de dados do aluno para identificar padrões, preferências e estilos de aprendizagem individuais que podem ser usados �����
Temp

## 7. Few-shot prompting
Incluímos exemplos para orientar o padrão da resposta.

In [7]:
prompt = """
Tarefa: classificar sentimento como POSITIVO ou NEGATIVO.

Texto: "Entrega rápida e produto excelente." -> POSITIVO
Texto: "Não funcionou e o suporte foi ruim." -> NEGATIVO
Texto: "Gostei, mas a bateria dura pouco." ->
""".strip()

print(generate_text(prompt, max_new_tokens=8, temperature=0.2, top_p=0.9))

Tarefa: classificar sentimento como POSITIVO ou NEGATIVO.

Texto: "Entrega rápida e produto excelente." -> POSITIVO
Texto: "Não funcionou e o suporte foi ruim." -> NEGATIVO
Texto: "Gostei, mas a bateria dura pouco." -> NEUTRO (MISTO)


## 8. Mini base de documentos
Vamos criar um conjunto pequeno de textos para simular uma base de conhecimento.

In [8]:
documents = [
    "RAG combina recuperação de documentos com geração para reduzir alucinações.",
    "Transformers usam atenção para modelar dependências longas em sequências.",
    "LLMs são treinados para prever o próximo token em grandes corpora de texto.",
    "RLHF ajusta modelos com feedback humano para seguir instruções.",
    "A avaliação de LLMs exige métricas automáticas e revisão humana."
]

## 9. Recuperação com TF-IDF
Usamos TF-IDF para buscar os documentos mais próximos da pergunta.

In [9]:
vectorizer = TfidfVectorizer()
X_docs = vectorizer.fit_transform(documents)

def retrieve(query, k=2):
    q_vec = vectorizer.transform([query])
    sims = cosine_similarity(q_vec, X_docs)[0]
    top_idx = sims.argsort()[::-1][:k]
    return [(documents[i], sims[i]) for i in top_idx]

query = "Por que RAG ajuda a reduzir alucinações?"
retrieved = retrieve(query, k=2)
retrieved

[('RAG combina recuperação de documentos com geração para reduzir alucinações.',
  np.float64(0.5970165324755593)),
 ('A avaliação de LLMs exige métricas automáticas e revisão humana.',
  np.float64(0.0))]

## 10. Geração ancorada (RAG simples)
Criamos um prompt que obriga o modelo a usar apenas o contexto recuperado.

In [10]:
context = "".join([f"- {doc}" for doc, _ in retrieved])

prompt = f"""
Responda usando apenas o contexto abaixo.

Contexto:
{context}

Pergunta: {query}
Resposta:
""".strip()

print('Sem contexto:')
print(generate_text(query, max_new_tokens=60, temperature=0.7, top_p=0.9))

print('Com contexto:')
print(generate_text(prompt, max_new_tokens=80, temperature=0.3, top_p=0.9))

Sem contexto:
Por que RAG ajuda a reduzir alucinações?
A Pesquisa Avançada em Gênero (RAG) é um projeto financiado pela National Science Foundation (NSF). O principal objetivo do projeto é desenvolver uma plataforma de computação quântica para análise estatística e aprendizado de máquina. No entanto, os pesquisadores descobriram recentemente que o algoritmo utilizado na pesquisa pode ajudar a
Com contexto:
Responda usando apenas o contexto abaixo.

Contexto:
- RAG combina recuperação de documentos com geração para reduzir alucinações.- A avaliação de LLMs exige métricas automáticas e revisão humana.

Pergunta: Por que RAG ajuda a reduzir alucinações?
Resposta: RAG ajuda a reduzir as alucinações, combinando a recuperação do documento com a geração automática, reduzindo assim os efeitos negativos da falta de compreensão contextual.


## 11. Observações finais
Este notebook mostra que LLMs geram texto fluente, mas a qualidade depende do prompt e do contexto. O uso de RAG melhora a confiabilidade, mas não elimina a necessidade de validação humana.